# Train PhoBertConceptStateTagger trên Colab

Notebook này chỉ lo phần môi trường (clone code, gắn dữ liệu/model từ Drive, cài thư viện) —
logic train thật sự nằm trong repo (`src/extraction/phobert_concept_state_tagger.py`,
`scripts/concept_state/train_phobert_concept_state_tagger.py`), đồng bộ qua git.

**Trước khi chạy:**
1. Đổi `DRIVE_ROOT` bên dưới nếu bạn để dữ liệu ở thư mục Drive khác.
2. Đã upload sẵn `data/concept_state/relations.csv` vào `DRIVE_ROOT/data/concept_state/relations.csv`
   trên Drive (sinh file này bằng `python -m scripts.concept_state.build_concept_state_relations`
   từ `data/llm_relations/relations.csv`, không chỉnh sửa tay qua Excel/Sheets — dễ làm hỏng
   quoting/field CSV).
3. Chọn Runtime > Change runtime type > GPU trước khi chạy (nếu có GPU free trên Colab).


## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 2. Clone / pull code từ git

Repo public, không cần token. Nếu đã clone từ lần trước, cell này sẽ `git pull` thay vì clone lại.

In [2]:
REPO_URL = "https://github.com/thnghia-ctu/CausalGraph.git"
BRANCH = "v3"
REPO_DIR = "/content/CausalGraph"

import os

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    !git clone -b $BRANCH $REPO_URL $REPO_DIR
    %cd $REPO_DIR


Cloning into '/content/CausalGraph'...
remote: Enumerating objects: 559, done.
remote: Counting objects: 100% (559/559), done.
remote: Compressing objects: 100% (396/396), done.
remote: Total 559 (delta 273), reused 420 (delta 139), pack-reused 0 (from 0)
Receiving objects: 100% (559/559), 1.70 MiB | 5.20 MiB/s, done.
Resolving deltas: 100% (273/273), done.
/content/CausalGraph


## 3. Gắn `data/` và `models/` vào Drive

Hai thư mục này bị `.gitignore`, không nằm trong git — clone xong sẽ trống hoặc không tồn tại.
Symlink sang Drive để dữ liệu và checkpoint được giữ lại qua các session, không cần copy tay
mỗi lần mở lại Colab.

In [3]:
DRIVE_ROOT = "/content/drive/MyDrive/CausalGraph"

import os

os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models", exist_ok=True)

!rm -rf {REPO_DIR}/data {REPO_DIR}/models
!ln -s {DRIVE_ROOT}/data {REPO_DIR}/data
!ln -s {DRIVE_ROOT}/models {REPO_DIR}/models

!ls -la {REPO_DIR}/data/concept_state/ 2>/dev/null || echo "Chưa có data/concept_state/relations.csv trên Drive — upload trước khi train."


total 505
-rw------- 1 root root 516203 Aug  4 12:24 relations.csv


## 4. Cài thư viện

In [4]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.2/387.2 kB 25.0 MB/s eta 0:00:00
   ━━━━

## 5. (Tuỳ chọn) Đăng nhập Hugging Face Hub

Cần nếu muốn push model đã train lên Hub ở bước cuối. Token tạo tại
https://huggingface.co/settings/tokens (quyền write).

In [5]:
from huggingface_hub import notebook_login

notebook_login()


## 6. Train

Checkpoint được lưu định kỳ vào `models/concept_state_tagger/` (= Drive, qua symlink ở bước 3).
Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần chạy lại cell này — `PhoBertConceptStateTagger.fit`
tự resume từ checkpoint gần nhất thay vì train lại từ đầu.

In [6]:
!python -m scripts.concept_state.train_phobert_concept_state_tagger


config.json: 100% 557/557 [00:00<00:00, 2.35MB/s]
vocab.txt: 100% 895k/895k [00:00<00:00, 12.6MB/s]
bpe.codes: 100% 1.14M/1.14M [00:00<00:00, 13.6MB/s]
tokenizer.json: 100% 3.13M/3.13M [00:00<00:00, 25.1MB/s]

pytorch_model.bin: downloading bytes:  67% 364M/543M [00:02<00:00, 183MB/s, 31.7MB/s  ]
pytorch_model.bin: reconstructing file:  74% 402M/543M [00:02<00:00, 153MB/s]
pytorch_model.bin: downloading bytes: 100% 366M/366M [00:03<00:00, 117MB/s, 32.4MB/s  ]
pytorch_model.bin: reconstructing file: 100% 543M/543M [00:03<00:00, 173MB/s, 49.0MB/s  ]

model.safetensors: downloading bytes:   0% 0.00/543M [00:00<?, ?B/s]

Loading weights: 100% 197/197 [00:00<00:00, 18682.66it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.decoder

## 7. (Tuỳ chọn) Push model lên Hugging Face Hub

Lưu bản train xong lên Hub để có version history riêng cho model, tách khỏi git,
và tải lại được từ máy local hoặc lần train sau mà không cần train lại.

In [7]:
from src.extraction.phobert_concept_state_tagger import PhoBertConceptStateTagger
from configs.config import CONCEPT_STATE_TAGGER_MODEL_DIR

HF_REPO_ID = "thnghia-ctu/vi-concept-state-tagger"  # đổi thành repo của bạn

tagger = PhoBertConceptStateTagger.load(CONCEPT_STATE_TAGGER_MODEL_DIR / "final")
tagger.push_to_hub(HF_REPO_ID)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...09n2td5/model.safetensors:   0%|          |  997kB /  538MB            